## Train and evaluate ML models on satellite image embeddings. 

This notebook facilitates geographic labelling, search of the embedding vectors, and lightweight machine learning with the embedding vectors as inputs. 

**Notebook outline** 

* **Embeddings:** Use a parquet file (embeddings + centroids in one table) or DuckDB assets (a `.db` plus a centroids parquet). BYOE. The mapper links lat/lon points to embedding vectors for labeling and ML. 

* **Labeler:** This is an iPyLeaflet map that allows interactive labeling of positive and negative data samples. It will also display reference data layers.

* **Nearest-neighbor search:** Akin to Earth Index quick search. Given a few positively and negatively labeled points, it will search the embedding vector space for other similar examples.

* **Training:** Code to or generate GeoJSON sets of positively or negatively labeled lat/lon points, and then to train and evaluate simple ML models. 

* **Run**: Code to run a model or ensemble of models across a whole vector embeddings set.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / 'src'))

import json
from datetime import datetime
import duckdb
import joblib
import geopandas as gpd
import ipyleaflet as ipyl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from IPython.display import display
import ipywidgets as ipyw

from embedding_store import from_dataframe, from_duckdb, get_annoy_index
from ui import BASEMAP_TILES, GeoLabeler
import alt_workflow_ml_utils as ml_utils

%load_ext autoreload
%autoreload 2

In [ ]:
# Paths: data dir and parquet (embeddings + centroids), or use DuckDB assets.
DATA_PATH = Path('/path/to/my/working/directory/')
PARQUET_PATH = DATA_PATH / 'embeddings.parquet'

# Alternative: DuckDB assets (centroids parquet + .db). Uncomment to use.
# CENTROID_PARQUET_PATH = DATA_PATH / 'centroids.parquet'
# DUCKDB_PATH = DATA_PATH / 'embeddings.db'

BOUNDARY_PATH = DATA_PATH / 'boundary.geojson'
BOUNDARY = gpd.read_file(BOUNDARY_PATH)


### Embeddings

In [ ]:
# 1. Read parquet and inspect columns so you can set id_column explicitly.
gdf = gpd.read_parquet(PARQUET_PATH)
display(gdf.head())

# Alternative: DuckDB assets. Load centroids, inspect (index.name or columns), then set id_column explicitly.
# centroid_gdf = gpd.read_parquet(CENTROID_PARQUET_PATH)
# display(centroid_gdf.head()); print("Index name:", centroid_gdf.index.name, "Columns:", list(centroid_gdf.columns))
# duckdb_con = duckdb.connect(str(DUCKDB_PATH))

In [ ]:
# 2. Set id_column to the column name you see for the id (e.g. "tile_id"), or None if there is no id column (ordinal 0,1,2,... will be used).
id_column = "tile_id"  # or None

In [ ]:
# 3. Build mapper
embeddings = from_dataframe(gdf, geometry_col="geometry", id_column=id_column)

# Alternative: DuckDB assets.
# embeddings = from_duckdb(centroid_gdf, duckdb_con, table_name="embeddings", id_column=id_column)

print(f"{len(embeddings.gdf)} embedding vectors loaded")

## Map labeling tool

In [ ]:
# Optional custom basemap URL. A map button will cycle through a custom basemap plus the defaults ui.BASEMAP_TILES.
kwin_tile2024 = 'https://tiles.earthindex.ai/v1/tiles/sentinel2-yearly-mosaics/2024-01-01/2025-01-01/rgb/{z}/{x}/{y}.webp'

labeler = GeoLabeler(embeddings.gdf, geojson_path=BOUNDARY_PATH, baselayer_url=kwin_tile2024, save_dir=DATA_PATH / "sampling_data")

label = ipyw.Label(); display(label)  

def handle_mouse_move(**kwargs):
    lat, lon = kwargs.get('coordinates')
    label.value = f'Lat/lon: {lat:.4f}, {lon:.4f}'

labeler.map.on_interaction(handle_mouse_move)

In [ ]:
# Optional reference points
with open(DATA_PATH / 'GardenCity_feedlotpolys_cattle_pens-cleaned.geojson') as f:
    est_layer = ipyl.GeoJSON(
        name="ref", data=json.load(f), style={'color': '#0cfbe4', 'opacity': 1, 'fillOpacity': 0, 'weight': 1})
    labeler.map.add_layer(est_layer)

In [ ]:
# Remove reference layer
for layer in labeler.map.layers:
    if hasattr(layer, 'name') and layer.name == 'ref':
        labeler.map.remove_layer(layer)
        break

## Quick search

Nearest-neighbor search in embedding space. The Annoy index is built from in-memory embeddings and saved to disk on first run; later runs load it from disk. Then we form a query vector from your positive/negative labels, search for similar points, clip to the boundary, and show them on the map.

In [ ]:
# Build index from in-memory vectors if not on disk; otherwise load.
ANNOY_INDEX_PATH = DATA_PATH / 'embeddings.ann'
if ANNOY_INDEX_PATH.exists():
    dim = int(input("Embedding dimension for existing index: "))
    annoy_index = get_annoy_index(ANNOY_INDEX_PATH, dim=dim)
else:
    vecs = embeddings.get_vectors(embeddings.gdf[embeddings.id_column])
    annoy_index = get_annoy_index(ANNOY_INDEX_PATH, vectors=vecs, n_trees=10)

In [ ]:
pos = embeddings.gdf.iloc[labeler.pos_indices]
neg = embeddings.gdf.iloc[labeler.neg_indices]

pos_vec = np.mean(embeddings.get_vectors(pos[embeddings.id_column]), axis=0)                
if len(neg) > 0:
    neg_vec = np.mean(embeddings.get_vectors(neg[embeddings.id_column]), axis=0) 
else:
    neg_vec = np.zeros(pos_vec.shape)

# Default query vector math, feel free to experiment with alternatives
query_vector = 2 * pos_vec - neg_vec

In [ ]:
n_nbors = 50
nbors = annoy_index.get_nns_by_vector(query_vector.astype(np.float32), n_nbors, include_distances=True)
detections = labeler.gdf.iloc[nbors[0]]

# This is required for lasso mode labeling of search results 
labeler.detection_gdf = detections[['geometry']]

labeler.update_layer(labeler.points, json.loads(detections.geometry.to_json()))

print(f'{len(labeler.pos_indices)} positives, {len(labeler.neg_indices)} negatives, {len(detections)} detections')

In [ ]:
# Another chance to save the pos/neg labels if not using the map Save Data button:
labeler.save_dataset()

In [ ]:
# Optional: save detections to the same directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
path = labeler.save_dir / f"ANNsearch_{timestamp}.geojson"
detections.to_file(path, driver="GeoJSON")
print(f"Saved detections to {path}")

In [ ]:
# Clear map
labeler.update_layer(labeler.points, {})

## Model training

#### Optional, one time: randomly sample points from the embeddings dataframe to build a negative dataset. This technique assumes that target objects are sparse geographically and allows a measure of noise in the dataset.

In [ ]:
n_samples = 5000
random_neg = embeddings.gdf.sample(n_samples, random_state=42)

In [ ]:
# Optional: clean up random samples which lie within known positive objects:
known_polys = gpd.read_file('/path/to/known_positive_polygons')
joined = gpd.sjoin(known_polys.loc[:, ['geometry']], random_neg, how="inner", op="contains")
random_neg = random_neg.drop(joined['index_right'])
print(f'After dropping from known positive regions, {len(random_neg)} samples remain.') 

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
path = labeler.save_dir / f"random_negatives_{timestamp}.geojson"
random_neg.to_file(path, driver="GeoJSON")
print(f"Saved to {path}")

### Training data

In [ ]:
pos_files = ['sampling_data/positive_points_20260303_175119.geojson',
             'sampling_data/positive_points_20260303_175303.geojson',
            ]

pos = pd.concat([gpd.read_file(DATA_PATH / f) for f in pos_files])
pos['geometry'] = pos['geometry'].centroid

# Note: it's essential not to use points from outside the embeddings AOI
pos = gpd.clip(pos, BOUNDARY)
pos = pos.drop(columns=[c for c in pos.columns if c != 'geometry']).reset_index(drop=True)

pos['int_class'] = 1
pos

In [ ]:
neg_files = ['sampling_data/negative_points_20260303_175303.geojson', 
             'sampling_data/random_negatives_20260303_181124.geojson'
            ]
neg = pd.concat([gpd.read_file(DATA_PATH / f) for f in neg_files])
neg['geometry'] = neg['geometry'].centroid
neg['int_class'] = 0
neg

In [ ]:
# Consider replicating or subsampling pos or neg sets for class balance 
training_points = pd.concat([pos, neg]).reset_index(drop=True)

In [ ]:
training_points[embeddings.id_column] = embeddings.map_points(training_points)
train, val = train_test_split(training_points, train_size=0.8, random_state=42)
display(train.int_class.value_counts())
train_X = embeddings.get_vectors(train[embeddings.id_column])
train_Y = train['int_class'].reset_index(drop=True)

In [ ]:
display(train_X.head())
train_Y.head()

#### Model

In [ ]:
# Either instantiate a new model:
#layer_sizes = (64, 16)
#model = MLPClassifier(hidden_layer_sizes=layer_sizes, n_iter_no_change=40, max_iter=1000, verbose=True)
model = LogisticRegression(max_iter=1000)

In [ ]:
# Or reload saved model, w/ warm_start for further training:
model_basepath = 'models/MLP64-16_2024-11-22T02:37.joblib'
model = joblib.load(DATA_PATH / model_basepath)
model.warm_start = True

# Disable early stopping and set a manual limit on training epochs
model.max_iter = 10
model.tol = 0
model.n_iter_no_change = 10000

In [ ]:
model.fit(train_X, train_Y)

#### Validation

In [ ]:
# F1 vs threshold (needs y_true and probs; threshold only used for curve sweep)
val_Y = val['int_class'].reset_index(drop=True)
val_probs, _ = ml_utils.predict_df(val, embeddings, model, threshold=0.5)
f1, _ = ml_utils.f1_curve(val_Y, val_probs)

In [ ]:
# Pick a threshold with reference to the F1 curve above
# (requires some trial and error)
threshold = 0.985
val_Y = val['int_class'].reset_index(drop=True)
probs, y_pred = ml_utils.predict_df(val, embeddings, model, threshold=threshold)
scores = ml_utils.score(y_pred, val_Y)

In [ ]:
pr, _ = ml_utils.prec_rec_curve(val_Y, val_probs)

In [ ]:
model_dir = DATA_PATH / 'models'
model_dir.mkdir(parents=True, exist_ok=True)

now = datetime.today().isoformat()[:16]
model_path = model_dir / f'{model.__class__.__name__}_{now}.joblib'
print(f'Model saved to: {model_path}')
joblib.dump(model, model_path)

output_specs = {
    'inputs': pos_files + neg_files, 
    'threshold': str(threshold), 
    'scores': scores, 
}

with open(model_dir / f"{model_path.stem}_specs.json", 'w') as f:
    json.dump(output_specs, f, indent=4)

In [ ]:
# Optional save of evaluation curves:
f1.savefig(model_dir / f"{model_path.stem}_F1.png")
pr.savefig(model_dir / f"{model_path.stem}_PR.png")

## Inference

In [ ]:
model_basepath = 'models/LogisticRegression_2026-03-04T00:41.joblib'
model = joblib.load(DATA_PATH / model_basepath)

threshold = 0.5 # Adjust this according to your determination above

# patch_width (m): width of the square patch aroundeach detection for polygon outputs. For 32-pixel Sentinel-2 patches use 320.
patch_width = 320

inference_dir = DATA_PATH / 'inference'
inference_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
detections = ml_utils.get_detections(embeddings, model, threshold=threshold, boundary_path=BOUNDARY_PATH)
polys = ml_utils.detections_to_rectpolys(embeddings, detections, patch_width=patch_width)
print(f"{len(detections)} detections; {len(polys)} polygons")

In [ ]:
# Generally I prefer output polygons, but the point detections make for a smaller file, so sometimes useful: 
# detections.to_file(inference_dir/ f'detections_Thresh{threshold}{Path(model_basepath).stem}.geojson')
polys.to_file(inference_dir / f'rectpolys_Thresh{threshold}{Path(model_basepath).stem}.geojson')

### Ensembling

In [ ]:
# Relax somewhat the single-model thresholds, because you will rethreshold later:
models = {
    'models/LogisticRegression1.joblib': 0.7, 
    'models/LogisticRegression2.joblib': 0.5,
    'models/LogisticRegression3.joblib': 0.55,
}

ensemble_dets = []
for path, threshold in models.items():
    model = joblib.load(DATA_PATH / path)
    detections = ml_utils.get_detections(embeddings, model, threshold=threshold, boundary_path=BOUNDARY_PATH)
    print(f"{path}: {len(detections)} detections")
    ensemble_dets.append(detections)

In [ ]:
dets = pd.concat(ensemble_dets, ignore_index=True)
if 'level_0' in dets.columns:
    dets = dets.drop(columns=['level_0'])
    
# Group for voting 
# Check for no counts greater than the number of models, which would indicate duplicate embedding patches
grouped = dets.drop(columns=['geometry']).groupby(by=embeddings.id_column).agg(['count', 'mean']).reset_index()
grouped.probability['count'].value_counts()

In [ ]:
req_votes = 3
id_column = embeddings.id_column  # e.g. 'tile_id'
voted = grouped[grouped.probability['count'] >= req_votes].copy()

# Build single-level DataFrame so merge works (left had MultiIndex columns)
id_col = voted[id_column].iloc[:, 0] if isinstance(voted[id_column], pd.DataFrame) else voted[id_column]
prob_mean = voted['probability']['mean']
voted = pd.DataFrame({id_column: id_col.values, 'probability': prob_mean.values})

# Restore geometry from embeddings (one geometry per id)
voted = voted.merge(embeddings.gdf[[id_column, 'geometry']], on=id_column, how='left')
voted = gpd.GeoDataFrame(voted, geometry='geometry')
print(f"{len(voted)} detections after voting")
voted.head()

In [ ]:
# patch_width (m): width of the square patch aroundeach detection for polygon outputs. For 32-pixel Sentinel-2 patches use 320.
patch_width = 320

voted_polys = ml_utils.detections_to_rectpolys(embeddings, voted, patch_width=patch_width)
voted_polys.head()

In [ ]:
now = datetime.today().isoformat()[:16]

#voted.to_file(inference_dir / f"detections_Ensemble_Voted{req_votes}of{len(models)}models{now}.geojson")
ensemble_outpath = inference_dir / f"rectpolysEnsemble_Voted{req_votes}of{len(models)}models{now}.geojson"
voted_polys.to_file(ensemble_outpath)

with open(inference_dir / f"{ensemble_outpath.stem}_models.json", 'w') as f:
    json.dump(models, f, indent=4)